In [ ]:
import os
from dotenv import load_dotenv
import databento as db

load_dotenv()
db_api_key = os.getenv("DATABENTO_API_KEY")

client = db.Historical(db_api_key)

try:
    cost = client.metadata.get_cost(
        dataset="GLBX.MDP3",
        schema="ohlcv-1m",
        stype_in="continuous",
        symbols=["NQ.v.0"],
        start="2026-11-17",
        end="2026-11-26",
    )
    print(cost, 'USD')
except db.BentoClientError as e:
    print(f"Error: {e}")


In [ ]:
try:
    db_data = client.timeseries.get_range(
        dataset="GLBX.MDP3",
        schema="ohlcv-1m",
        stype_in="continuous",
        symbols=["NQ.v.0"],
        start="2026-11-17",
        end="2026-11-26",
    )
    db_data = db_data.to_df()
except db.BentoClientError as e:
    print(f"Error: {e}")

- Fact: Databento flags 2026-11-22 as degraded
- Impact: store a quality flag per day, check if the model is sensitive to it

In [ ]:
display(db_data.head())
display(db_data.columns)
display(db_data.dtypes) 
display(db_data.index.tz) 
display(len(db_data))

- Fact: 9 columns, prices are float64, index in UTC
- Impact: keep prices, volume, index, instrument_id drop symbols, rtype, publisher_id
- Fact: 9 660 rows, expected 9 660 rows
- Impact: don't assume every minute exists (holidays, short sessions); add a dbt test comparing expected vs received bars per day

In [ ]:
db_ny = db_data.tz_convert("America/New_York")
db_ny.head()

In [ ]:
display(db_ny.groupby(db_ny.index.hour).size())
display(db_ny.groupby(db_ny.index.day_of_week).size())

- Fact: no bars between 17:00 and 18:00 New York time (daily halt), no Saturday, Sunday only from 18:00
- Fact: bar counts per calendar day are uneven, because a NQ session runs from 18:00 the previous day to 17:00 (New York time)
- Impact: add a `session_date` column following the CME rule (a session is named after the day it ends), never group by calendar date
- Impact: always convert to New York time with pandas (`tz_convert`), never with a fixed offset, because of daylight saving time

- Fact: the 9:30-16:00 window is the regular session (RTH) of the cash market; NQ trades almost 24h on Globex
- Impact: organise data by Globex session, compute the target and the gap on the RTH window

In [ ]:
display(db_ny.groupby(db_ny.index.hour)["volume"].sum())

volume_by_hour = db_ny.groupby(db_ny.index.hour)["volume"].sum()
rth_share = volume_by_hour.loc[9:15].sum() / volume_by_hour.sum()
rth_share

- Fact: ~77% of the volume trades between 9:00 and 16:00 New York time, peak at 10:00 (~15x an average night hour)
- Fact: volume jumps at 8:00, when US macro data is released (8:30)
- Impact: pre-market features come from low-volume hours, so they are noisier; prefer ranges and returns over single prices
- Impact: macro release days (8:30) need a flag, they change the pre-market behaviour

In [ ]:
day = db_ny.loc["2026-11-24"]
morning = day.between_time("09:30", "11:29")
print("nrows: ", len(morning))
max_high = morning["high"].max()
print("max_high: ", max_high)
min_low = morning["low"].min()
print("min_low: ", min_low)
open_value = morning["open"].iloc[0]

amplitude = (max_high - min_low)/open_value*100
print("amplitude: ", amplitude)

- Fact: morning range on 2026-11-24 = 1.94% (high 26,509.75, low 25,998.00), same high and low as TradingView
- Impact: target rule = (max high - min low) / open, on bars starting between 9:30 and 11:29 New York time, open = open of the 9:30 bar

In [ ]:
try:
    cost = client.metadata.get_cost(
        dataset="GLBX.MDP3",
        schema="ohlcv-1m",
        stype_in="continuous",
        symbols=["NQ.v.0"],
        start="2026-09-08",
        end="2026-09-19",
    )
    print(cost, 'USD')
except db.BentoClientError as e:
    print(f"Error: {e}")


In [ ]:
try:
    db_data = client.timeseries.get_range(
        dataset="GLBX.MDP3",
        schema="ohlcv-1m",
        stype_in="continuous",
        symbols=["NQ.v.0"],
        start="2026-09-08",
        end="2026-09-19",
    )
    db_data = db_data.to_df()
except db.BentoClientError as e:
    print(f"Error: {e}")

In [ ]:
db_data["instrument_id_changed"] = db_data["instrument_id"] != db_data["instrument_id"].shift(1, fill_value=db_data["instrument_id"].iloc[0])
roll_time = db_data[db_data["instrument_id_changed"] == True]
roll_time

In [ ]:
roll_pos =db_data.index.get_loc(roll_time.index[0])
around_positions= db_data.iloc[roll_pos-1:roll_pos+1]

around_positions

In [ ]:
diff = around_positions["open"].values[1] - around_positions["close"].values[0]
diff_percent = diff / around_positions["close"].values[0] * 100
print("diff: ", diff)
print("diff_percent: ", diff_percent)

- Fact: NQ.v.0 rolled from the September to the December contract on 2026-09-16 at 00:00 UTC (20:00 New York, in the middle of a Globex session)
- Fact: price jump of +293.25 points (+1.01%), about half a morning range; the last hours before the switch come from a nearly empty contract (5 lots vs 150)
- Impact: flag roll sessions; never compute a gap or a return across two different instrument_id
- Impact: the roll does not happen on a session boundary, so a roll session mixes two contracts

In [ ]:
import yfinance as yf

dat = yf.download(["^NDX", "^VXN"], start="2026-10-27", end="2026-11-27", auto_adjust=False, )

display(dat.index.tz)

dat.head()

- Fact: yfinance returns wide columns (field, ticker), daily index without timezone; Adj Close = Close for indices
- Fact: ^VXN volume is always 0, ^NDX volume is the sum of its components' shares, not comparable with NQ volume
- Impact: reshape yfinance data to long format (one row per ticker and date), round prices to 2 decimals
- Impact: drop volume for ^NDX and ^VXN; auto_adjust has no effect on indices

In [ ]:
try:
    nq_close = client.timeseries.get_range(
        dataset="GLBX.MDP3",
        schema="ohlcv-1m",
        stype_in="continuous",
        symbols=["NQ.v.0"],
        start="2026-11-24T20:55:00",
        end="2026-11-24T21:00:00",
    )
    nq_close = nq_close.to_df()
except db.BentoClientError as e:
    print(f"Error: {e}")


In [ ]:
nq_close

In [ ]:
close_11_24_yfinance = dat.loc["2026-11-24"]["Close"]["^NDX"]
close_11_24_db= nq_close["close"].values[-1]
print("close_11_24_yfinance: ", close_11_24_yfinance)
print("close_11_24_db: ", close_11_24_db)
diff_close = close_11_24_db - close_11_24_yfinance
diff_close_percent = diff_close / close_11_24_db * 100
print("diff_close: ", diff_close)
print("diff_close_percent: ", diff_close_percent)

- Fact: on 2026-11-24 at 16:00 New York, NQ (26,284.25) traded +70.08 points (+0.27%) above ^NDX (26,214.17): the basis
- Fact: the basis shrinks to 0 at expiry and grows with time to expiry (the roll jump was ~1% for 3 more months)
- Impact: NQ is the reference instrument (target, round levels); ^NDX is only an input feature
- Impact: never mix NQ and ^NDX prices in one feature without accounting for the basis